# Extra — News sentiment for your dashboard (Gemini)

**Why this earns its place:** it adds a *sentiment panel* next to a stock's price.
The LLM reads recent headlines and returns a score from -1 (very negative) to +1
(very positive). It never makes a trade — it adds *context*.

Runs in **MOCK mode** with no key. Add a Gemini key to use the real model.

## Setup
Free key: https://aistudio.google.com/apikey — then set `GEMINI_API_KEY`.

In [ ]:
import os
API_KEY = os.environ.get('GEMINI_API_KEY')  # None -> mock mode
USE_MOCK = API_KEY is None
print('MOCK mode' if USE_MOCK else 'using Gemini')

## Some headlines (replace with a real feed later)

In [ ]:
headlines = {
  'COMI': ['Bank posts record quarterly profit', 'Analysts raise price target'],
  'HRHO': ['Firm faces regulatory review', 'Weak trading revenue reported'],
  'SWDY': ['Wins large infrastructure contract', 'Expands into new market'],
}
for k,v in headlines.items(): print(k, '->', v)

## Score sentiment
Ask the model for a single number per stock. In mock mode we fake a plausible score.

In [ ]:
import json, re

PROMPT = ('You are a financial sentiment rater. Given headlines about a stock, '
          'reply with ONLY a JSON object like {"score": 0.4} where score is '
          'between -1 (very negative) and 1 (very positive).')

def score_real(items):
    import google.generativeai as genai
    genai.configure(api_key=API_KEY)
    model = genai.GenerativeModel('gemini-1.5-flash')
    text = PROMPT + '\nHeadlines: ' + ' | '.join(items)
    resp = model.generate_content(text)
    m = re.search(r'\{.*\}', resp.text, re.S)
    return float(json.loads(m.group())['score'])

def score_mock(items):
    pos = sum(w in ' '.join(items).lower() for w in ['record','profit','wins','raise','expands'])
    neg = sum(w in ' '.join(items).lower() for w in ['review','weak','faces','cut','loss'])
    return round((pos-neg)/max(pos+neg,1), 2)

score = score_mock if USE_MOCK else score_real
scores = {k: score(v) for k,v in headlines.items()}
print(scores)

## Show it as a panel
This is the dashboard-ready output — a colored bar per stock.

In [ ]:
import matplotlib.pyplot as plt
names=list(scores); vals=[scores[n] for n in names]
colors=['green' if v>0 else 'red' if v<0 else 'gray' for v in vals]
plt.figure(figsize=(7,3)); plt.bar(names, vals, color=colors); plt.axhline(0,color='k',lw=.6)
plt.ylim(-1,1); plt.title('Headline sentiment'); plt.show()

## Wire into the dashboard
Add a backend endpoint `/sentiment` returning `scores`, and a frontend panel that
renders these bars. Now your product shows *why* a stock is moving, not just that
it moved.